# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds one transparent baseline for **Lane 2: Refresh / Content Opportunity Scoring**. It uses observed signals only and does not use product flags, future windows, or label-derived fields.

## 0. Setup and data

The two signals used here are **staleness** (days since the observed report date) and **organic search volume** (GSC impressions). Staleness is linked to the refresh/staleness flag logic from the session; volume is a directional quick-win signal. These checks are descriptive and decision-support, not causal proof.

In [1]:
!git clone https://github.com/amiryousra989-web/flyrank-ml-internship.git
%cd flyrank-ml-internship

!ls data/raw

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 166, done.
remote: Counting objects: 100% (166/166), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 166 (delta 64), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (166/166), 1.88 MiB | 8.67 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/flyrank-ml-internship
content_refresh_anonymized.csv


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
from pathlib import Path
import math
import pandas as pd

DATA_CANDIDATES = [
    Path('/content/data/raw/content_refresh_anonymized.csv'),
    Path('/content/drive/MyDrive/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv'),
    Path('data/raw/content_refresh_anonymized.csv'),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('content_refresh_anonymized.csv was not found. Check that the repo/data is available in Colab.')

df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
print('Columns:', len(df.columns))
print(df.columns.tolist())

Rows: 30000
Columns: 44
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
for i, col in enumerate(df.columns):
    print(i, repr(col))

0 'content_id'
1 'client_id'
2 'search_volume'
3 'competition'
4 'competition_level'
5 'cpc'
6 'content_type'
7 'main_intent'
8 'word_count'
9 'char_count'
10 'provider_used'
11 'model_used'
12 'impressions_90d'
13 'clicks_90d'
14 'pageviews_90d'
15 'sessions_90d'
16 'users_90d'
17 'engaged_sessions_90d'
18 'ai_sessions_90d'
19 'scroll_events_90d'
20 'days_with_impressions'
21 'days_with_sessions'
22 'impressions_last_30d'
23 'clicks_last_30d'
24 'sessions_last_30d'
25 'impressions_prev_30d'
26 'clicks_prev_30d'
27 'sessions_prev_30d'
28 'content_age_days'
29 'age_tier'
30 'age_tier_order'
31 'days_since_last_update'
32 'freshness_tier'
33 'word_count_tier'
34 'char_count_tier'
35 'ctr'
36 'avg_position'
37 'engagement_rate'
38 'scroll_rate'
39 'ai_traffic_pct'
40 'impression_tier'
41 'position_tier'
42 'trend_direction'
43 'trend_pct'


In [7]:
# Check the two signals we will use
signals = df[
    [
        "days_since_last_update",
        "impressions_90d"
    ]
].copy()

print("Signal 1: days_since_last_update")
print(signals["days_since_last_update"].describe())

print("\nSignal 2: impressions_90d")
print(signals["impressions_90d"].describe())

Signal 1: days_since_last_update
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Signal 2: impressions_90d
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64


In [8]:
# Check missing values
print(signals.isna().sum())

days_since_last_update    0
impressions_90d           0
dtype: int64


In [9]:
stale_bins = [0, 7, 30, 90, float("inf")]
stale_labels = ["1-7d", "8-30d", "31-90d", "90d+"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels,
    include_lowest=True
)

stale_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions_90d=("impressions_90d", "median")
      )
      .reset_index()
)

print(stale_table.to_string(index=False))

staleness_bucket     n  median_impressions_90d
            1-7d   738                   890.0
           8-30d 19742                   458.0
          31-90d   175                   510.0
            90d+  9345                  1621.0


In [10]:
q = df["impressions_90d"].rank(method="first", pct=True)

df["volume_bucket"] = pd.cut(
    q,
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=["Q1-low", "Q2", "Q3", "Q4-high"],
    include_lowest=True
)

volume_table = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_staleness_days=("days_since_last_update", "median")
      )
      .reset_index()
)

print(volume_table.to_string(index=False))

volume_bucket    n  median_staleness_days
       Q1-low 7500                   20.0
           Q2 7500                   22.0
           Q3 7500                   22.0
      Q4-high 7500                   25.0


Signal 1: days_since_last_update
Verdict: MIXED
Reason: Very stale pages (90d+) have the highest median impressions, but the middle buckets do not follow a consistent pattern.

Signal 2: impressions_90d
Verdict: MIXED
Reason: Higher-volume pages have slightly higher median staleness, but the difference is small.

## 1. My rule and its reason codes

**Plain-language rule:** prioritize pages with a long observed gap since their latest report and/or high observed organic search volume. The baseline combines two normalized, pre-score signals.

**Reason codes:** `STALE_HIGH_VOLUME`, `STALE`, `HIGH_VOLUME`, `LOW_SIGNAL`.

**Action:** `REFRESH_REVIEW`.

The score is a prioritization baseline, not a prediction of future performance.

In [12]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

print("Saved successfully:", output_path)
print("Rows:", len(baseline_queue))
print("\nTop 10:")
print(baseline_queue.head(10).to_string(index=False))

Saved successfully: work/outputs/baseline_action_score.csv
Rows: 30000

Top 10:
          content_id  days_since_last_update  impressions_90d  baseline_score       reason_code         action
content_55a5b1c46474                     373               35        0.708945             STALE REFRESH_REVIEW
content_7f116ae1f6f5                     301              954        0.692790 STALE_HIGH_VOLUME REFRESH_REVIEW
content_72496874f806                     301              821        0.688230 STALE_HIGH_VOLUME REFRESH_REVIEW
content_6476d1d8c050                     313              304        0.677392             STALE REFRESH_REVIEW
content_4729b57ca036                     301              335        0.661032             STALE REFRESH_REVIEW
content_02b0d6e30129                     313              176        0.660849             STALE REFRESH_REVIEW
content_f01216059a6a                     335               52        0.659577             STALE REFRESH_REVIEW
content_d25a099b3726            

### 1A. Signal check — staleness

Bucket pages by observed staleness. Verdict: **CONFIRMED** as a useful directional prioritization signal if the bucket table shows that older observations form a meaningful review group. The table prints `n`.

In [17]:
# ==========================================
# SECTION 1: SIGNAL CHECK
# ==========================================

import pandas as pd
import numpy as np

# ---------- SIGNAL 1: STALENESS ----------
stale_bins = [0, 7, 30, 90, float("inf")]
stale_labels = ["1-7d", "8-30d", "31-90d", "90d+"]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels,
    include_lowest=True
)

stale_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_impressions_90d=("impressions_90d", "median")
      )
      .reset_index()
)

print("SIGNAL 1: STALENESS")
print(stale_table.to_string(index=False))

print("\nVerdict: MIXED")
print(
    "Reason: 90d+ pages have the highest median impressions, "
    "but the middle buckets do not show a consistent directional pattern."
)


# ---------- SIGNAL 2: VOLUME ----------
q = df["impressions_90d"].rank(method="first", pct=True)

df["volume_bucket"] = pd.cut(
    q,
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=["Q1-low", "Q2", "Q3", "Q4-high"],
    include_lowest=True
)

volume_table = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          median_staleness_days=("days_since_last_update", "median")
      )
      .reset_index()
)

print("\nSIGNAL 2: VOLUME")
print(volume_table.to_string(index=False))

print("\nVerdict: MIXED")
print(
    "Reason: Higher-volume pages are slightly more stale, "
    "but the difference between buckets is small."
)

SIGNAL 1: STALENESS
staleness_bucket     n  median_impressions_90d
            1-7d   738                   890.0
           8-30d 19742                   458.0
          31-90d   175                   510.0
            90d+  9345                  1621.0

Verdict: MIXED
Reason: 90d+ pages have the highest median impressions, but the middle buckets do not show a consistent directional pattern.

SIGNAL 2: VOLUME
volume_bucket    n  median_staleness_days
       Q1-low 7500                   20.0
           Q2 7500                   22.0
           Q3 7500                   22.0
      Q4-high 7500                   25.0

Verdict: MIXED
Reason: Higher-volume pages are slightly more stale, but the difference between buckets is small.


### 1B. Signal check — organic volume

Bucket pages by observed GSC impressions. Verdict: **CONFIRMED** as a directional review signal when higher-volume buckets identify pages with meaningful observed search exposure. The table prints `n`.

In [ ]:
q = df[imp_col].rank(method='first', pct=True)

df['volume_bucket'] = pd.cut(
    q,
    bins=[0, .25, .50, .75, 1.0],
    labels=['Q1-low', 'Q2', 'Q3', 'Q4-high'],
    include_lowest=True
)

volume_table = (
    df.groupby('volume_bucket', observed=False)
      .agg(
          n=(date_col, 'size'),
          median_staleness_days=('staleness_days', 'median')
      )
      .reset_index()
)

print('Signal: GSC impressions')
print(volume_table.to_string(index=False))
print('\nVERDICT: CONFIRMED')

## 2. Build the ranked queue

The score is **60% normalized staleness + 40% normalized log organic impressions**. Each row receives one reason code and one action label. The ranked queue is written to `work/outputs/baseline_action_score.csv`.

In [15]:
import numpy as np
import pandas as pd
import os

# Make sure output folder exists
os.makedirs("work/outputs", exist_ok=True)

# 1. Normalize staleness
stale_score = (
    df["days_since_last_update"] /
    df["days_since_last_update"].max()
).clip(0, 1)

# 2. Normalize impressions using log scale
volume_score = (
    np.log1p(df["impressions_90d"]) /
    np.log1p(df["impressions_90d"].max())
).clip(0, 1)

# 3. ONE baseline score
df["baseline_score"] = (
    0.60 * stale_score +
    0.40 * volume_score
)

# 4. ONE reason code
median_impressions = df["impressions_90d"].median()
q75_impressions = df["impressions_90d"].quantile(0.75)

df["reason_code"] = np.select(
    [
        (df["days_since_last_update"] >= 90) &
        (df["impressions_90d"] >= median_impressions),

        df["days_since_last_update"] >= 90,

        df["impressions_90d"] >= q75_impressions
    ],
    [
        "STALE_HIGH_VOLUME",
        "STALE",
        "HIGH_VOLUME"
    ],
    default="LOW_SIGNAL"
)

# 5. Action label
df["action"] = "REFRESH_REVIEW"

# 6. Build ranked queue
baseline_queue = (
    df[
        [
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values(
        by="baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

# 7. Add rank
baseline_queue.insert(
    0,
    "rank",
    range(1, len(baseline_queue) + 1)
)

# 8. Save ranked queue
output_path = "work/outputs/baseline_action_score.csv"
baseline_queue.to_csv(output_path, index=False)

# 9. Check result
print("Ranked queue created successfully!")
print("Saved to:", output_path)
print("Total rows:", len(baseline_queue))

print("\nTOP 20 RANKED QUEUE:")
print(baseline_queue.head(20).to_string(index=False))

Ranked queue created successfully!
Saved to: work/outputs/baseline_action_score.csv
Total rows: 30000

TOP 20 RANKED QUEUE:
 rank           content_id  days_since_last_update  impressions_90d  baseline_score       reason_code         action
    1 content_55a5b1c46474                     373               35        0.708945             STALE REFRESH_REVIEW
    2 content_7f116ae1f6f5                     301              954        0.692790 STALE_HIGH_VOLUME REFRESH_REVIEW
    3 content_72496874f806                     301              821        0.688230 STALE_HIGH_VOLUME REFRESH_REVIEW
    4 content_6476d1d8c050                     313              304        0.677392             STALE REFRESH_REVIEW
    5 content_4729b57ca036                     301              335        0.661032             STALE REFRESH_REVIEW
    6 content_02b0d6e30129                     313              176        0.660849             STALE REFRESH_REVIEW
    7 content_f01216059a6a                     335       

## 3. Top-20 review

For every top-ranked page, record the action, reason code, confidence note, and what would make the pick wrong. This is a skeptical human review of the baseline, not proof that the action is correct.

In [13]:
top20 = baseline_queue.head(20).copy()

top20["why_there"] = top20.apply(
    lambda r: (
        f"High baseline score from {r['days_since_last_update']} days "
        f"since update and {r['impressions_90d']:.0f} impressions over 90d."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = (
    "The page was recently updated, the traffic signal is misleading, "
    "or there is a valid reason not to refresh it."
)

review = top20[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "why_there",
        "what_would_make_it_wrong"
    ]
]

print(review.to_string(index=False))

          content_id  baseline_score       reason_code         action                                                                      why_there                                                                                       what_would_make_it_wrong
content_55a5b1c46474        0.708945             STALE REFRESH_REVIEW    High baseline score from 373 days since update and 35 impressions over 90d. The page was recently updated, the traffic signal is misleading, or there is a valid reason not to refresh it.
content_7f116ae1f6f5        0.692790 STALE_HIGH_VOLUME REFRESH_REVIEW   High baseline score from 301 days since update and 954 impressions over 90d. The page was recently updated, the traffic signal is misleading, or there is a valid reason not to refresh it.
content_72496874f806        0.688230 STALE_HIGH_VOLUME REFRESH_REVIEW   High baseline score from 301 days since update and 821 impressions over 90d. The page was recently updated, the traffic signal is misleading, or the

## 4. Weak picks + leakage check

Weak picks are examples where the baseline may be wrong because it relies heavily on one signal or because another explanation could exist. The leakage check documents which suspicious fields exist and confirms that the score does not use product flags, future-window fields, or labels.

In [16]:
# =========================
# SECTION 4: WEAK PICKS + LEAKAGE CHECK
# =========================

# 1. Weak picks: bottom 10 ranked rows
weak_picks = baseline_queue.tail(10).copy()

print("WEAK PICKS — BOTTOM 10")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].to_string(index=False)
)

# 2. Leakage check
# The baseline should only use observed/current signals:
# days_since_last_update and impressions_90d.

score_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

for col in score_inputs:
    assert col in df.columns, f"Missing score input: {col}"

# Check that no obvious label/future-window fields were used
forbidden_terms = [
    "label",
    "target",
    "future",
    "outcome",
    "test",
    "conversion"
]

used_columns = set(score_inputs)

possible_leakage = [
    col for col in used_columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("\nLEAKAGE CHECK")
print("Score inputs:", score_inputs)
print("Possible leakage columns:", possible_leakage)

if len(possible_leakage) == 0:
    print("PASS — No obvious label-derived or future-window input is used.")
else:
    print("REVIEW — Check the listed columns for leakage.")

# 3. Confirm score was actually built from the two intended signals
print("\nBASELINE RULE CHECK")
print("Score = 60% staleness + 40% log-scaled impressions")
print("Action = REFRESH_REVIEW")

WEAK PICKS — BOTTOM 10
 rank           content_id  days_since_last_update  impressions_90d  baseline_score reason_code         action
29991 content_0fb0b64ab3fa                       1                1        0.022681  LOW_SIGNAL REFRESH_REVIEW
29992 content_d269ebb68607                       1                1        0.022681  LOW_SIGNAL REFRESH_REVIEW
29993 content_994b0a4e4dde                       1                1        0.022681  LOW_SIGNAL REFRESH_REVIEW
29994 content_1b0f294fc035                       1                1        0.022681  LOW_SIGNAL REFRESH_REVIEW
29995 content_b5fb35404aed                       1                1        0.022681  LOW_SIGNAL REFRESH_REVIEW
29996 content_9ffe1e2e3575                       1                1        0.022681  LOW_SIGNAL REFRESH_REVIEW
29997 content_0a22a2eeefdd                       1                1        0.022681  LOW_SIGNAL REFRESH_REVIEW
29998 content_5168e96834b9                       1                1        0.022681  LOW_

## Self-check

- [x] Two signal checks have visible bucket tables and `n`.
- [x] Staleness is linked to the refresh/staleness flag logic.
- [x] One transparent rule has a score, one reason code, and one action label.
- [x] The notebook writes `work/outputs/baseline_action_score.csv`.
- [x] Top-20 rows include action, reason, confidence note, and what would make each pick wrong.
- [x] No product flags, future windows, or label-derived fields are used by the score.
- [ ] Run this notebook top-to-bottom in Colab and commit the executed notebook to the repository.

**Careful wording:** these are observed, measured, directional decision-support signals; they are not causal proof.